# CIDA Role Pre-Training v2 (Foundation Edition)
This notebook contains the complete pipeline for pretraining **RoleCIDA** to foundation-grade specialization using `microsoft/deberta-v3-base` (768 dimensions) and official Stanford/NYU NLI datasets (bypassing HF datasets API completely).

## Key Upgrades
1. **DeBERTa-v3-base**: Strong encoder for logical/semantic relationships.
2. **Direct Academic Downloads**: Downloads SNLI from Stanford and MultiNLI from NYU via direct ZIP URLs to bypass HuggingFace datasets library issues.
3. **Connected Gradient Graph**: Fixed backprop flow through anchors.
4. **CIDAInferenceEngine**: High-level wrapper class to run predictions and get structured outputs immediately after training.


## 0. Setup and Dependencies

In [ ]:
!pip install torchdiffeq transformers sentence-transformers scikit-learn -q
!git clone https://github.com/Kairatzh/CIDA-plugin.git 2>/dev/null || (cd CIDA-plugin && git pull)
%cd CIDA-plugin

## 1. Role Identity Layer v2

In [ ]:
!mkdir -p cida_role_pretrain_v2
%%writefile cida_role_pretrain_v2/role_identity.py
"""
RoleIdentityLayer v2 вЂ” Fixed: anchors now feed INTO CIDAPlugin.
+ Role-Conditioned Messaging
+ Contrastive Message Loss (InfoNCE)
+ Role-specific time constants П„
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from collections import defaultdict


class RoleIdentityLayer(nn.Module):
    """
    Learnable role anchors injected DIRECTLY into agent init states.
    
    v2 fixes:
    - Anchors are projected into d_hidden (not d_emb) so they match CIDAPlugin
    - forward() returns (B, M, d_hidden) ready for plugin.s_init
    - anchor_diversity_loss() computed on RAW anchors, not blended vectors
    """
    ROLE_PROTOTYPES = {
        0: [  # Prosecutor
            'The evidence clearly demonstrates this is true.',
            'Multiple studies confirm this conclusion beyond doubt.',
            'This is definitively supported by the data.',
            'The facts unambiguously point to this being correct.',
            'There is overwhelming support for this position.',
        ],
        1: [  # Defender
            'However, this argument fails to account for key exceptions.',
            'The opposing evidence suggests we should reconsider.',
            'This claim is not supported when examined carefully.',
            'There are significant flaws in this line of reasoning.',
            'Alternative explanations are equally or more plausible.',
        ],
        2: [  # Skeptic
            'It is unclear whether this conclusion is warranted.',
            'The evidence is mixed and difficult to interpret.',
            'We cannot be certain given the available information.',
            'This remains an open question with no clear answer.',
            'Further investigation is needed before drawing conclusions.',
        ],
        3: [  # Integrator
            'Taking all perspectives into account, the most balanced view is.',
            'Weighing the evidence from all sides leads to this synthesis.',
            'Both arguments have merit and the truth likely lies between them.',
            'Integrating these viewpoints gives us a more complete picture.',
            'The consensus emerging from all perspectives suggests.',
        ],
    }

    def __init__(self, d_emb, d_hidden, n_agents=4, embed_fn=None):
        super().__init__()
        self.n_agents = n_agents
        self.d_hidden = d_hidden

        # Initialize anchors from prototypes if embed_fn provided
        if embed_fn is not None:
            anchors = []
            for i in range(n_agents):
                proto_vecs = embed_fn(self.ROLE_PROTOTYPES[i])
                anchor = proto_vecs.mean(0)
                anchors.append(anchor)
            self.anchors = nn.Parameter(torch.stack(anchors))  # (M, d_emb)
        else:
            self.anchors = nn.Parameter(torch.randn(n_agents, d_emb) * 0.02)

        # Project anchors from d_emb -> d_hidden (matches CIDAPlugin)
        self.anchor_proj = nn.Sequential(
            nn.Linear(d_emb, d_hidden),
            nn.LayerNorm(d_hidden),
        )
        # Input projection (same as CIDAPlugin.input_proj)
        self.input_proj = nn.Sequential(
            nn.Linear(d_emb, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.SiLU(),
        )
        # Learnable blend factor per agent
        self.blend_alpha = nn.Parameter(torch.full((n_agents,), 0.3))

    def forward(self, x):
        """
        x: (B, d_emb) вЂ” raw embedding from encoder
        Returns: (B, M, d_hidden) вЂ” role-conditioned initial states for agents
        """
        B = x.size(0)
        h = self.input_proj(x)  # (B, d_hidden)
        h_exp = h.unsqueeze(1).expand(B, self.n_agents, -1)  # (B, M, d_hidden)

        anchors_norm = F.normalize(self.anchors, p=2, dim=-1)
        anchors_h = self.anchor_proj(anchors_norm)  # (M, d_hidden)
        anchors_exp = anchors_h.unsqueeze(0).expand(B, -1, -1)  # (B, M, d_hidden)

        alpha = self.blend_alpha.sigmoid().view(1, self.n_agents, 1)  # (1, M, 1)
        combined = (1 - alpha) * h_exp + alpha * anchors_exp
        return combined  # (B, M, d_hidden)

    def get_anchor_embeds(self):
        """Return projected anchors for use in losses (detached from input)."""
        with torch.no_grad():
            a = F.normalize(self.anchors, p=2, dim=-1)
        return self.anchor_proj(a)  # (M, d_hidden)

    def anchor_diversity_loss(self):
        """Orthogonality on RAW anchors (fix: no input data contamination)."""
        a = F.normalize(self.anchors, p=2, dim=-1)  # (M, d_emb)
        gram = a @ a.T
        eye = torch.eye(self.n_agents, device=a.device)
        return ((gram - eye) ** 2).mean()

    def anchor_similarity_matrix(self):
        a = F.normalize(self.anchors, p=2, dim=-1)
        return (a @ a.T).detach().cpu()


class RoleConditionedMessageFormulator(nn.Module):
    """
    v2: Role anchor is concatenated to agent state BEFORE message generation.
    This forces each agent to produce role-characteristic messages at every round.
    """
    def __init__(self, d_hidden, num_classes, d_message, n_agents=4, anonymous=True):
        super().__init__()
        self.anonymous = anonymous
        self.n_agents = n_agents
        # Role embedding added to message input
        self.role_embed = nn.Embedding(n_agents, d_hidden)
        if anonymous:
            in_dim = d_hidden + num_classes + 1 + d_hidden  # [e; b; u; role]
        else:
            in_dim = d_hidden * 2 + num_classes + 1 + d_hidden  # [s; e; b; u; role]
        self.proj = nn.Sequential(
            nn.Linear(in_dim, d_message),
            nn.LayerNorm(d_message),
            nn.SiLU(),
            nn.Linear(d_message, d_message),
        )

    def forward(self, s, e, b, u):
        B, M, _ = s.shape
        role_ids = torch.arange(M, device=s.device).unsqueeze(0).expand(B, -1)
        role_emb = self.role_embed(role_ids)  # (B, M, d_hidden)
        if self.anonymous:
            concat = torch.cat([e, b, u, role_emb], dim=-1)
        else:
            concat = torch.cat([s, e, b, u, role_emb], dim=-1)
        return self.proj(concat)


class ContrastiveMessageLoss(nn.Module):
    """
    InfoNCE loss: a classifier should identify WHO sent a message
    from its latent representation. Forces agents to develop unique
    communication signatures.
    """
    def __init__(self, d_message, n_agents=4, temperature=0.07):
        super().__init__()
        self.classifier = nn.Linear(d_message, n_agents)
        self.temperature = temperature

    def forward(self, m_all):
        """
        m_all: list of (B, M, d_message) per round
        Returns: scalar loss
        """
        if not m_all:
            return torch.tensor(0.0, requires_grad=True)
        losses = []
        for m_t in m_all:
            B, M, d = m_t.shape
            m_flat = m_t.reshape(B * M, d)  # (B*M, d)
            labels = torch.arange(M, device=m_t.device).unsqueeze(0).expand(B, -1).reshape(-1)
            logits = self.classifier(m_flat) / self.temperature
            losses.append(F.cross_entropy(logits, labels))
        return torch.stack(losses).mean()


class RoleTimeDynamics(nn.Module):
    """
    Role-specific time constants П„ for deliberation dynamics.
    
    Skeptic: high П„ (resistant to persuasion, "viscous")
    Integrator: low П„ (rapidly adapts, "fluid")
    Prosecutor/Defender: medium П„
    """
    def __init__(self, n_agents=4, d_hidden=256):
        super().__init__()
        # Learnable base П„ per agent, initialized by role
        init_tau = torch.tensor([1.0, 1.0, 2.0, 0.5])  # P, D, S, I
        self.log_tau = nn.Parameter(torch.log(init_tau[:n_agents]))
        # Data-dependent modulation
        self.tau_mod = nn.Sequential(
            nn.Linear(d_hidden, n_agents),
            nn.Sigmoid(),
        )

    def forward(self, s_mean):
        """
        s_mean: (B, d_hidden) вЂ” mean agent state
        Returns: (B, M) вЂ” per-agent time constants
        """
        base_tau = self.log_tau.exp().unsqueeze(0)  # (1, M)
        mod = self.tau_mod(s_mean)  # (B, M) in [0, 1]
        return base_tau * (0.5 + mod)  # П„ в€€ [0.5В·base, 1.5В·base]


## 2. Model & Losses v2

In [ ]:
%%writefile cida_role_pretrain_v2/model_and_losses.py
"""
RoleCIDA v2 вЂ” Model wrapper + Losses + Training engine.
All fixes applied: connected graph, clean losses, EMA, gradient verification.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, copy
from collections import defaultdict

from role_identity import (
    RoleIdentityLayer,
    RoleConditionedMessageFormulator,
    ContrastiveMessageLoss,
    RoleTimeDynamics,
)


class RoleCIDA(nn.Module):
    """
    v2: Role anchors are INJECTED into CIDAPlugin's initial agent states.
    
    Data flow:
        x (B, d_emb) 
          в†’ RoleIdentityLayer в†’ s_init (B, M, d_hidden)
          в†’ CIDAPlugin uses s_init as starting agent states
          в†’ Role-Conditioned Messages at each round
          в†’ Consensus
    """
    def __init__(self, d_emb, d_hidden, num_classes, plugin_cls, plugin_cfg,
                 embed_fn=None):
        super().__init__()
        self.d_emb = d_emb
        self.d_hidden = d_hidden
        self.num_classes = num_classes

        # Role identity layer (produces s_init for agents)
        self.role_layer = RoleIdentityLayer(
            d_emb=d_emb, d_hidden=d_hidden, n_agents=4, embed_fn=embed_fn
        )
        # CIDA plugin (we disable its internal PerspectiveProjector)
        plugin_cfg.use_perspective_projector = False
        self.plugin = plugin_cls(plugin_cfg)
        # Override plugin's message formulator with role-conditioned version
        self.role_msg = RoleConditionedMessageFormulator(
            d_hidden=d_hidden,
            num_classes=num_classes,
            d_message=plugin_cfg.d_message,
            n_agents=4,
            anonymous=plugin_cfg.anonymous_messages,
        )
        # Contrastive message loss
        self.msg_contrast = ContrastiveMessageLoss(
            d_message=plugin_cfg.d_message, n_agents=4
        )
        # Role time dynamics
        self.time_dynamics = RoleTimeDynamics(n_agents=4, d_hidden=d_hidden)

    def forward(self, x):
        """
        x: (B, d_emb) вЂ” pre-computed embedding
        """
        # 1) Role-conditioned initial states (FIX: anchors now IN the graph)
        s_init = self.role_layer(x)  # (B, M, d_hidden)

        # 2) Inject s_init into plugin by monkey-patching perspective projector
        #    We pass raw x to plugin but override the initial state
        out = self._forward_with_role_init(x, s_init)

        # 3) Add role-specific outputs for loss computation
        out['anchor_div_loss'] = self.role_layer.anchor_diversity_loss()
        out['msg_contrast_loss'] = self.msg_contrast(out.get('m_all', []))
        out['tau'] = self.time_dynamics(x[:, :self.d_hidden]
                                         if x.size(-1) >= self.d_hidden
                                         else self.role_layer.input_proj(x))
        out['anchor_sim'] = self.role_layer.anchor_similarity_matrix()
        return out

    def _forward_with_role_init(self, x, s_init):
        """Run CIDAPlugin but replace agent initialization with s_init."""
        from cida_plugin.agent import AgentState, apply_role_priors_batched
        cfg = self.plugin.config
        B = x.size(0)
        M = cfg.num_agents
        K = cfg.num_classes

        # Project input (for evidence extraction)
        h_cls = self.plugin.input_proj(x)  # (B, d_hidden)

        # Use role-conditioned s_init instead of PerspectiveProjector
        s_t = s_init  # (B, M, d_hidden) вЂ” FROM RoleIdentityLayer

        seq_len = 1
        e_init = torch.zeros(B, M, cfg.d_hidden, device=x.device)
        r_init = torch.zeros_like(e_init)

        if cfg.multi_label:
            b_init = torch.full((B, M, K), 0.5, device=x.device)
        else:
            b_init = torch.zeros(B, M, K, device=x.device)

        state_t = AgentState(
            s=s_t, b=b_init,
            u=torch.ones(B, M, 1, device=x.device),
            p=torch.zeros(B, M, seq_len, device=x.device),
            e=e_init,
            alpha=torch.ones(B, M, K, device=x.device),
        )

        # Initial update
        state_t = self.plugin.updater(state_t, r_init, e_init)
        state_t.b = apply_role_priors_batched(
            state_t.b, M, multi_label=cfg.multi_label,
            positive_class_idx=cfg.positive_class_idx,
        )

        # Deliberation loop with role-conditioned messages
        b_all, u_all, m_all = [], [], []
        for t in range(cfg.max_rounds):
            # Evidence (no seq_output в†’ use state as evidence)
            p_new = torch.ones(B, M, seq_len, device=x.device) / seq_len
            e_new = state_t.s
            state_t.p = p_new
            state_t.e = e_new

            # Role-conditioned messages (v2 improvement)
            m_new = self.role_msg(state_t.s, state_t.e, state_t.b, state_t.u)

            # Communication
            r_new = self.plugin.communication(
                state_t.s, m_new, state_t.b, state_t.e,
                sparse_k=cfg.sparse_comm_k
            )
            gate = self.plugin.comm_gate(state_t.s)
            r_new = gate * r_new
            if self.training and cfg.comm_dropout > 0:
                r_new = F.dropout(r_new, p=cfg.comm_dropout, training=True)

            # Update
            state_t = self.plugin.updater(state_t, r_new, e_new)
            state_t.b = apply_role_priors_batched(
                state_t.b, M, multi_label=cfg.multi_label,
                positive_class_idx=cfg.positive_class_idx,
            )

            b_all.append(state_t.b)
            u_all.append(state_t.u)
            m_all.append(m_new)

        # Consensus
        p_final, uncertainty, disagreement = self.plugin.consensus(
            state_t.b, state_t.u
        )
        from cida_plugin.agent import compute_role_orthogonality_loss
        L_orth = compute_role_orthogonality_loss(state_t.s)

        return {
            'p_final': p_final,
            'b_all': b_all,
            'u_all': u_all,
            'm_all': m_all,
            'uncertainty': uncertainty,
            'disagreement': disagreement,
            'L_orth': L_orth,
            'rounds_used': cfg.max_rounds,
            'state_final': state_t,
        }


class RolePreTrainLossV2(nn.Module):
    """
    v2: Behavioral loss with fixes:
    - Anchor diversity on RAW anchors (not blended)
    - Consensus coherence properly scaled
    - Added contrastive message loss integration
    """
    def __init__(self, n_classes=3):
        super().__init__()
        self.K = n_classes

    def forward(self, b_all, arg_roles, anchor_div_loss, msg_contrast_loss,
                stage, anneal=1.0):
        if not b_all:
            return torch.tensor(0.0, requires_grad=True), {}

        eps = 1e-7
        B = arg_roles.size(0)
        K = self.K
        device = arg_roles.device

        pro_mask = (arg_roles == 0)
        unc_mask = (arg_roles == 1)
        con_mask = (arg_roles == 2)

        total_loss = torch.tensor(0.0, device=device)
        info = defaultdict(float)
        max_ent = math.log(K)

        round_w = torch.linspace(0.4, 1.0, len(b_all), device=device)
        round_w = round_w / round_w.sum()

        for rnd, (b_t, w_r) in enumerate(zip(b_all, round_w)):
            _, M, _ = b_t.shape
            b_t = b_t[:, :, :K].clamp(eps, 1-eps)
            b_t = b_t / b_t.sum(-1, keepdim=True)

            agent_losses = []
            for i in range(min(M, 4)):
                b_i = b_t[:, i, :]
                conf = b_i.max(dim=1).values
                ent = -(b_i * b_i.log()).sum(dim=1)
                b_pro = b_i[:, 0]
                b_con = b_i[:, min(2, K-1)]

                if i == 0:  # PROSECUTOR
                    l_pro = -torch.log(b_pro[pro_mask] + eps).mean() if pro_mask.any() else torch.tensor(0., device=device)
                    l_con = torch.clamp(b_con[con_mask] - 0.5, min=0).mean() if con_mask.any() else torch.tensor(0., device=device)
                    l_unc = -torch.log(b_pro[unc_mask] + eps).mean() * 0.3 if unc_mask.any() else torch.tensor(0., device=device)
                    loss_i = l_pro + 0.3*l_con + 0.2*l_unc
                    info['prosecutor'] += loss_i.item()
                elif i == 1:  # DEFENDER
                    l_con = -torch.log(b_con[con_mask] + eps).mean() if con_mask.any() else torch.tensor(0., device=device)
                    l_pro = torch.clamp(b_pro[pro_mask] - 0.5, min=0).mean() if pro_mask.any() else torch.tensor(0., device=device)
                    l_unc = -torch.log(b_con[unc_mask] + eps).mean() * 0.3 if unc_mask.any() else torch.tensor(0., device=device)
                    loss_i = l_con + 0.3*l_pro + 0.2*l_unc
                    info['defender'] += loss_i.item()
                elif i == 2:  # SKEPTIC
                    l_unc = -ent[unc_mask].mean() / max_ent if unc_mask.any() else torch.tensor(0., device=device)
                    certain_mask = ~unc_mask
                    l_cert = torch.clamp(conf[certain_mask] - 0.7, min=0).mean() if certain_mask.any() else torch.tensor(0., device=device)
                    loss_i = l_unc + 0.4*l_cert
                    info['skeptic'] += loss_i.item()
                else:  # INTEGRATOR
                    y_c = arg_roles.clamp(0, K-1)
                    loss_i = F.nll_loss(b_i.log(), y_c)
                    info['integrator'] += loss_i.item()

                agent_losses.append(loss_i)

            # Consensus coherence
            b_mean = b_t.mean(dim=1, keepdim=True)
            integrator_b = b_t[:, 3:4, :]
            coherence = F.kl_div(
                (integrator_b + eps).log(), b_mean,  # FIX: no eps on target
                reduction='batchmean'
            )
            info['coherence'] += coherence.item()

            round_total = (torch.stack(agent_losses).mean()
                          + 0.05 * coherence * anneal)
            total_loss = total_loss + w_r * round_total

        n_rounds = len(b_all)
        # FIX: anchor diversity on RAW anchors (passed in, not computed from blended)
        total_loss = total_loss + 0.1 * anchor_div_loss
        info['anchor_div'] = anchor_div_loss.item()

        # NEW: contrastive message loss
        total_loss = total_loss + 0.05 * msg_contrast_loss * anneal
        info['msg_contrast'] = msg_contrast_loss.item()

        return total_loss, {k: v/n_rounds if k not in ('anchor_div','msg_contrast') else v
                           for k, v in info.items()}


class UncertaintyCalibrationLossV2(nn.Module):
    """v2: Fixed KL target normalization."""
    def forward(self, b_all, arg_roles, soft_labels=None):
        if not b_all:
            return torch.tensor(0.0, requires_grad=True), {}
        eps = 1e-7
        K = b_all[0].shape[-1]

        if soft_labels is None:
            y_oh = F.one_hot(arg_roles.clamp(0, K-1), K).float()
            alpha = 0.7
            soft = alpha * y_oh + (1-alpha)/K
            # FIX: ensure soft sums to 1.0 (no eps contamination)
            soft = soft / soft.sum(dim=-1, keepdim=True)
        else:
            soft = soft_labels

        b_final = b_all[-1][:, :, :K].clamp(eps, 1-eps)
        b_final = b_final / b_final.sum(-1, keepdim=True)

        losses = []
        for i in range(b_final.size(1)):
            b_i = b_final[:, i, :]
            if i == 2:  # Skeptic
                kl = F.kl_div((b_i+eps).log(), soft, reduction='batchmean')
                losses.append(kl)
            elif i == 3:  # Integrator
                kl = F.kl_div((b_i+eps).log(), soft, reduction='batchmean')
                losses.append(kl * 0.8)
            else:
                ce = F.nll_loss(b_i.log(), arg_roles.clamp(0, K-1))
                losses.append(ce * 0.5)

        loss = torch.stack(losses).mean()
        return loss, {'calib': loss.item()}


class EMAModel:
    """Exponential Moving Average of model weights for stable evaluation."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)

    def apply(self, model):
        """Temporarily apply EMA weights."""
        self._backup = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)

    def restore(self, model):
        """Restore original weights."""
        model.load_state_dict(self._backup)
        del self._backup


## 3. Foundation Training Script

In [ ]:
%%writefile cida_role_pretrain_v2/train_foundation.py
"""
CIDA Role Pre-Training v2 вЂ” Foundation-Grade Script.
- Encoder: microsoft/deberta-v3-base (768 dimensions)
- Datasets: Direct downloads of official SNLI (Stanford) & MultiNLI (NYU)
- Target: Production-ready checkpoint with zero HF dataset dependencies
"""
import sys, os, time, math, warnings, zipfile, urllib.request
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import defaultdict
warnings.filterwarnings('ignore')

# в”Ђв”Ђ Setup в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IS_AMP = DEVICE == 'cuda'
DTYPE = torch.bfloat16 if (DEVICE=='cuda' and torch.cuda.get_device_capability()[0]>=8) else torch.float16
torch.manual_seed(42); np.random.seed(42)
os.makedirs('ckpt_role_foundation', exist_ok=True)
os.makedirs('results_role_foundation', exist_ok=True)
D_EMB, D_HIDDEN = 768, 256 # DeBERTa-v3 has 768 dims

# в”Ђв”Ђ 1. Robust Downloaders (Bypassing HF datasets API) в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
def download_and_extract_nli(data_dir="nli_data"):
    os.makedirs(data_dir, exist_ok=True)
    snli_url = "https://nlp.stanford.edu/projects/snli/snli_1.0.zip"
    mnli_url = "https://cims.nyu.edu/~sbowman/multinli/multinli_1.0.zip"
    
    snli_zip = os.path.join(data_dir, "snli.zip")
    mnli_zip = os.path.join(data_dir, "mnli.zip")
    
    # Download SNLI if not exists
    if not os.path.exists(os.path.join(data_dir, "snli_1.0", "snli_1.0_train.txt")):
        print("Downloading SNLI from Stanford University (90MB)...")
        urllib.request.urlretrieve(snli_url, snli_zip)
        with zipfile.ZipFile(snli_zip, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(snli_zip)
        
    # Download MultiNLI if not exists
    if not os.path.exists(os.path.join(data_dir, "multinli_1.0", "multinli_1.0_train.txt")):
        print("Downloading MultiNLI from NYU (220MB)...")
        urllib.request.urlretrieve(mnli_url, mnli_zip)
        with zipfile.ZipFile(mnli_zip, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(mnli_zip)

def parse_nli_file(file_path, max_samples=150000):
    texts, labels = [], []
    label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
    with open(file_path, 'r', encoding='utf-8') as f:
        header = f.readline().strip().split('\t')
        s1_idx = header.index("sentence1")
        s2_idx = header.index("sentence2")
        lbl_idx = header.index("gold_label")
        
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) <= max(s1_idx, s2_idx, lbl_idx):
                continue
            lbl = parts[lbl_idx]
            if lbl in label_map:
                s1, s2 = parts[s1_idx], parts[s2_idx]
                if len(s1.strip()) > 5 and len(s2.strip()) > 5:
                    texts.append(f"{s1} | {s2}")
                    labels.append(label_map[lbl])
            if len(texts) >= max_samples:
                break
    return texts, labels

# в”Ђв”Ђ 2. DeBERTa-v3-base Encoder в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
from transformers import AutoTokenizer, AutoModel
print("Initializing DeBERTa-v3-base model...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
encoder = AutoModel.from_pretrained("microsoft/deberta-v3-base").to(DEVICE)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

@torch.no_grad()
def embed_texts(texts, batch_size=256):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        outputs = encoder(**inputs)
        # Mean pooling
        attention_mask = inputs['attention_mask'].unsqueeze(-1)
        embeddings = outputs.last_hidden_state * attention_mask
        embeddings = embeddings.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1e-9)
        vecs.append(embeddings.cpu())
    return torch.cat(vecs)

# в”Ђв”Ђ 3. Import CIDA + v2 modules в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
if '__file__' in globals():
    base_dir = os.path.dirname(os.path.abspath(__file__))
else:
    base_dir = os.path.abspath('.')
sys.path.insert(0, base_dir)
sys.path.insert(0, os.path.join(base_dir, '..'))
sys.path.insert(0, os.path.join(base_dir, 'cida_role_pretrain_v2'))

from cida_plugin import CIDAPlugin, CIDAPluginConfig
from cida_plugin.agent import RoleNames
from role_identity import RoleIdentityLayer, ContrastiveMessageLoss, RoleTimeDynamics
from model_and_losses import RoleCIDA, RolePreTrainLossV2, UncertaintyCalibrationLossV2, EMAModel

# в”Ђв”Ђ 4. Main script execution в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
def main():
    # Download & parse datasets
    download_and_extract_nli()
    print("Parsing NLI files...")
    snli_texts, snli_labels = parse_nli_file("nli_data/snli_1.0/snli_1.0_train.txt", max_samples=100000)
    mnli_texts, mnli_labels = parse_nli_file("nli_data/multinli_1.0/multinli_1.0_train.txt", max_samples=100000)
    
    all_texts = snli_texts + mnli_texts
    all_labels = snli_labels + mnli_labels
    
    print(f"Total texts parsed: {len(all_texts):,}")
    
    # Pre-compute DeBERTa-v3 embeddings
    print("Embedding texts with DeBERTa-v3-base...")
    t0 = time.time()
    embs = embed_texts(all_texts)
    roles = torch.tensor(all_labels, dtype=torch.long)
    print(f"Embeddings generated in {time.time()-t0:.1f}s. Shape: {embs.shape}")
    
    # Make Datasets and Loaders
    class EmbeddingDataset(Dataset):
        def __init__(self, embs, roles): self.embs, self.roles = embs, roles
        def __len__(self): return len(self.roles)
        def __getitem__(self, idx): return {'emb': self.embs[idx], 'role': self.roles[idx]}

    def make_loaders(embs, roles, val_ratio=0.05, bs=256):
        n = len(roles); nv = max(1000, int(n*val_ratio))
        idx = torch.randperm(n)
        tr = EmbeddingDataset(embs[idx[nv:]], roles[idx[nv:]])
        vl = EmbeddingDataset(embs[idx[:nv]], roles[idx[:nv]])
        kw = dict(batch_size=bs, num_workers=4, pin_memory=True, persistent_workers=True)
        return DataLoader(tr, shuffle=True, **kw), DataLoader(vl, shuffle=False, **kw)
        
    s1_trl, s1_vll = make_loaders(embs, roles, bs=256)
    
    # Configure CIDA with DeBERTa input size (768)
    cfg = CIDAPluginConfig(
        d_input=D_EMB, d_hidden=D_HIDDEN, d_message=D_HIDDEN,
        num_agents=4, num_classes=3, max_rounds=3, multi_label=False,
        num_attn_heads=8, role_prior_blend=0.40, lambda_cal=0.3, lambda_ac=0.1,
        min_disagreement=0.08, comm_dropout=0.15, use_perspective_projector=True,
        anonymous_messages=True, sparse_comm_k=2, use_dynamic_trust=True,
        trust_ema_gamma=0.9,
    )
    
    # Load model
    model = RoleCIDA(D_EMB, D_HIDDEN, 3, CIDAPlugin, cfg, embed_fn=None).to(DEVICE)
    role_loss_fn = RolePreTrainLossV2(n_classes=3)
    calib_loss_fn = UncertaintyCalibrationLossV2()
    
    optimizer = optim.AdamW([
        {'params': model.role_layer.parameters(), 'lr': 5e-4, 'weight_decay': 0.0},
        {'params': model.plugin.parameters(), 'lr': 2e-4, 'weight_decay': 1e-2},
        {'params': model.role_msg.parameters(), 'lr': 3e-4, 'weight_decay': 1e-3},
        {'params': model.msg_contrast.parameters(), 'lr': 3e-4, 'weight_decay': 0.0},
        {'params': model.time_dynamics.parameters(), 'lr': 1e-4, 'weight_decay': 0.0},
    ], fused=(DEVICE=='cuda'))
    
    STAGE_EPOCHS = {1: 4, 2: 3, 3: 3}
    total_steps = sum(STAGE_EPOCHS[s]*len(s1_trl) for s in [1, 2, 3])
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer,
        max_lr=[8e-4, 3e-4, 5e-4, 5e-4, 2e-4],
        total_steps=total_steps, pct_start=0.05, anneal_strategy='cos')
    ema = EMAModel(model, decay=0.999)
    
    def compute_asi(b_all):
        if not b_all: return 0.0
        b = b_all[-1]
        ents = [-(b[:,ag,:].clamp(1e-7,1)*b[:,ag,:].clamp(1e-7,1).log()).sum(1).mean().item()
                for ag in range(b.shape[1])]
        return float(np.std(ents))

    @torch.no_grad()
    def evaluate(mdl, loader, max_batches=80):
        mdl.eval()
        all_acc, all_dis, all_asi = [], [], []
        agent_conf, agent_ent = defaultdict(list), defaultdict(list)
        for i, batch in enumerate(loader):
            if i >= max_batches: break
            x, y = batch['emb'].to(DEVICE), batch['role'].to(DEVICE).clamp(0, 2)
            with torch.autocast(device_type='cuda' if DEVICE=='cuda' else 'cpu', dtype=DTYPE, enabled=IS_AMP):
                out = mdl(x)
            preds = out['p_final'][:, :3].argmax(1)
            all_acc.append((preds == y).float().mean().item())
            all_dis.append(out['disagreement'].mean().item())
            all_asi.append(compute_asi(out['b_all']))
            if out['b_all']:
                b = out['b_all'][-1]
                for ag in range(b.shape[1]):
                    bi = b[:, ag, :].clamp(1e-7, 1)
                    agent_conf[ag].append(bi.max(1).values.mean().item())
                    agent_ent[ag].append(-(bi*bi.log()).sum(1).mean().item())
        agents = {ag: {'conf': np.mean(agent_conf[ag]), 'ent': np.mean(agent_ent[ag])}
                  for ag in sorted(agent_conf)}
        return {'acc': np.mean(all_acc), 'disagreement': np.mean(all_dis),
                'asi': np.mean(all_asi), 'agents': agents}

    def train_epoch(mdl, loader, opt, sched, stage, anneal):
        mdl.train()
        total_loss, n = 0.0, 0
        scaler = torch.cuda.amp.GradScaler(enabled=(DTYPE==torch.float16))
        for batch in loader:
            x = batch['emb'].to(DEVICE, non_blocking=True)
            y = batch['role'].to(DEVICE, non_blocking=True).clamp(0, 2)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda' if DEVICE=='cuda' else 'cpu', dtype=DTYPE, enabled=IS_AMP):
                out = mdl(x)
                if stage in [1, 3]:
                    loss, info = role_loss_fn(out['b_all'], y,
                        out['anchor_div_loss'], out['msg_contrast_loss'], stage, anneal)
                else:
                    loss_r, _ = role_loss_fn(out['b_all'], y,
                        out['anchor_div_loss'], out['msg_contrast_loss'], 1, anneal)
                    loss_c, _ = calib_loss_fn(out['b_all'], y)
                    loss = 0.5 * loss_r + 0.5 * loss_c
                if out.get('L_orth') is not None:
                    loss = loss + 0.02 * out['L_orth']
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(mdl)
            total_loss += loss.item(); n += 1
        return total_loss / max(n, 1)
        
    best_asi = 0.0
    global_ep = 0
    history = []
    
    for stage in [1, 2, 3]:
        names = {1:'Role Identity', 2:'Uncertainty Calibration', 3:'Cross-Domain'}
        print(f'\n{"="*60}\n  STAGE {stage}: {names[stage]}\n{"="*60}')
        for ep in range(STAGE_EPOCHS[stage]):
            global_ep += 1
            anneal = ep / max(STAGE_EPOCHS[stage]-1, 1)
            t0 = time.time()
            avg_loss = train_epoch(model, s1_trl, optimizer, scheduler, stage, anneal)
            
            # Evaluate
            ema.apply(model)
            ev = evaluate(model, s1_vll)
            ema.restore(model)
            
            dt = time.time() - t0
            history.append({'stage':stage,'epoch':global_ep,'loss':avg_loss,'time':dt,
                            **{k:v for k,v in ev.items() if k!='agents'}})
            
            print(f'\nEp {global_ep}/{sum(STAGE_EPOCHS.values())} [{dt:.0f}s] '
                  f'loss={avg_loss:.4f} acc={ev["acc"]:.4f} '
                  f'dis={ev["disagreement"]:.4f} ASI={ev["asi"]:.4f}')
            
            for ag, s in ev['agents'].items():
                print(f'  {RoleNames.get(ag):<12}: conf={s["conf"]:.3f} ent={s["ent"]:.3f}')
            
            sim = model.role_layer.anchor_similarity_matrix()
            off = sim[~torch.eye(4, dtype=bool)].abs().mean().item()
            print(f'  Anchor off-diag: {off:.4f}')
            
            if ev['asi'] > best_asi:
                best_asi = ev['asi']
                torch.save({
                    'plugin': model.plugin.state_dict(),
                    'role_layer': model.role_layer.state_dict(),
                    'role_msg': model.role_msg.state_dict(),
                    'msg_contrast': model.msg_contrast.state_dict(),
                    'time_dynamics': model.time_dynamics.state_dict(),
                    'ema_shadow': ema.shadow,
                    'config': model.plugin.config.to_dict(),
                }, 'ckpt_role_foundation/best_foundation.pt')
                print(f'  вњ“ Best Saved (ASI={best_asi:.4f})')
                
    print("\nPre-training Complete! Checkpoint saved to: ckpt_role_foundation/best_foundation.pt")

# в”Ђв”Ђ 5. Production Inference Engine в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
class CIDAInferenceEngine:
    """
    Production-ready wrapper for CIDA role-conditioned deliberation.
    Accepts raw strings, runs DeBERTa encoder, and outputs consensus,
    disagreement, and agent thought-process trace.
    """
    def __init__(self, checkpoint_path, device=DEVICE):
        self.device = device
        # Load config and architecture
        ckpt = torch.load(checkpoint_path, map_location=device)
        cfg_dict = ckpt['config']
        cfg = CIDAPluginConfig(**cfg_dict)
        
        self.model = RoleCIDA(d_emb=cfg.d_input, d_hidden=cfg.d_hidden, 
                              num_classes=cfg.num_classes, plugin_cls=CIDAPlugin, 
                              plugin_cfg=cfg).to(device)
        self.model.load_state_dict(ckpt['ema_shadow'] if 'ema_shadow' in ckpt else ckpt['plugin'])
        self.model.eval()
        
        # Load DeBERTa encoder
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
        self.encoder = AutoModel.from_pretrained("microsoft/deberta-v3-base").to(device)
        self.encoder.eval()
        for p in self.encoder.parameters():
            p.requires_grad = False
            
    @torch.no_grad()
    def analyze(self, texts):
        """
        Analyze a list of texts.
        Returns detailed consensus metrics and agent beliefs per round.
        """
        # Embed
        inputs = self.tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(self.device)
        outputs = self.encoder(**inputs)
        attention_mask = inputs['attention_mask'].unsqueeze(-1)
        embeddings = outputs.last_hidden_state * attention_mask
        x = embeddings.sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1e-9)
        
        # Forward pass through RoleCIDA
        out = self.model(x)
        
        # Format results
        results = []
        for idx in range(len(texts)):
            trace = []
            for r_idx in range(len(out['b_all'])):
                beliefs = out['b_all'][r_idx][idx].cpu().tolist()
                trace.append({
                    "round": r_idx + 1,
                    "prosecutor_belief": beliefs[0],
                    "defender_belief": beliefs[1],
                    "skeptic_belief": beliefs[2],
                    "integrator_belief": beliefs[3]
                })
            
            results.append({
                "text": texts[idx],
                "final_prediction": out['p_final'][idx].cpu().tolist(),
                "predicted_class": int(out['p_final'][idx].argmax().item()),
                "uncertainty": float(out['uncertainty'][idx].item()),
                "disagreement": float(out['disagreement'][idx].item()),
                "deliberation_trace": trace
            })
        return results

if __name__ == '__main__':
    main()


## 4. Run Foundation Pretraining

In [ ]:
!python cida_role_pretrain_v2/train_foundation.py

## 5. Production Inference Demo
Use the pre-trained foundation CIDA model directly on custom texts:

In [ ]:
# Import engine from the script
import sys
sys.path.insert(0, './cida_role_pretrain_v2')
from train_foundation import CIDAInferenceEngine

# Initialize engine with the best checkpoint
engine = CIDAInferenceEngine('ckpt_role_foundation/best_foundation.pt')

# Real-world diagnostic/logical test texts
test_cases = [
    'The patient shows clear ground glass opacities in both lungs. | This indicates a severe case of viral pneumonia.',
    'The patient is coughing. | The patient definitely has tuberculosis.',
]

# Run deliberation
analyses = engine.analyze(test_cases)

for i, res in enumerate(analyses):
    print(f'\nCase {i+1}: "{res["text"]}"')
    print(f'  - Predicted Class: {res["predicted_class"]} (0=Entailment, 1=Neutral, 2=Contradiction)')
    print(f'  - Disagreement Score: {res["disagreement"]:.4f}')
    print(f'  - Uncertainty: {res["uncertainty"]:.4f}')
    print('  - Deliberation Trace (Integrator Consensus):')
    for r in res['deliberation_trace']:
        print(f'    Round {r["round"]}: P={r["prosecutor_belief"]}, D={r["defender_belief"]}, S={r["skeptic_belief"]}, I={r["integrator_belief"]}')
